# Week 3 — Data Cleaning & Transformation

**Dataset:** `carlist_100k_pw.csv` — 100,000 Malaysian used car listings scraped from carlist.my

This notebook cleans and transforms the raw dataset, then saves the result as `carlist_cleaned.csv`.

In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

RAW_CSV     = '../carlist_100k_pw.csv'
CLEANED_CSV = 'carlist_cleaned.csv'

## 1. Load Raw Data

In [2]:
df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape}')
print(f'Raw memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
display(df.head(3))

Shape: (100018, 11)
Raw memory usage: 85.0 MB


,id,title,year,price,monthly_payment,mileage,transmission,seller_type,location,condition,url
0,18990060,2026 BMW 218 1.5 Gran Sport Coupe - FREE WARRA...,2026,"RM 208,000","RM 2,697 / month",0 - 5K KM,Automatic,Sales Agent This 'Trusted Dealer' has a proven...,"Selangor, Shah Alam",USED,https://www.carlist.my/used-cars/2026-bmw-218-...
1,18974192,2017 Toyota Corolla Altis 1.8 G Sedan,2017,"RM 55,800",RM 723 / month,90 - 95K KM,Automatic,Dealer This 'Trusted Dealer' has a proven trac...,"Perak, Teluk Intan",USED,https://www.carlist.my/used-cars/2017-toyota-c...
2,18990035,2020 Nissan Serena 2.0 S-Hybrid High-Way Star ...,2020,"RM 69,800",RM 905 / month,85 - 90K KM,Automatic,Sales Agent This 'Trusted Dealer' has a proven...,"Selangor, Seri Kembangan",USED,https://www.carlist.my/used-cars/2020-nissan-s...


## 2. Profile Missing & Blank Values

Check every column for `NaN` values and empty strings before cleaning.

In [3]:
null_counts  = df.isnull().sum()
blank_counts = df.apply(lambda col: (col.astype(str).str.strip() == '').sum())

missing_report = pd.DataFrame({
    'null_count' : null_counts,
    'blank_count': blank_counts,
    'total_rows' : len(df)
})
missing_report['missing_pct'] = (
    (missing_report['null_count'] + missing_report['blank_count'])
    / missing_report['total_rows'] * 100
).round(2)

print('Missing value report:')
print(missing_report)

Missing value report:
                 null_count  blank_count  total_rows  missing_pct
id                        0            0      100018         0.00
title                     0            0      100018         0.00
year                      0            0      100018         0.00
price                     0            0      100018         0.00
monthly_payment         146            0      100018         0.15
mileage                   0            0      100018         0.00
transmission              0            0      100018         0.00
seller_type              45            0      100018         0.04
location                  0            0      100018         0.00
condition                 0            0      100018         0.00
url                       0            0      100018         0.00


## 3. Clean & Transform

Steps applied:
1. `seller_type` — strip boilerplate text after the label
2. `price` — remove discount prefix (e.g. `-4% RM 105,800`), strip `RM` and commas → float
3. `monthly_payment` — strip `RM` and `/ month`; blanks → `NaN`
4. `mileage` — extract lower-bound km as integer (`0 - 5K KM` → 0, `90 - 95K KM` → 90000)
5. `location` — split `State, City` into two columns
6. `year` — ensure integer type
7. Fill remaining blanks with `N/A`

In [4]:
def clean_price(s):
    """Strip RM prefix/discount notation and return float."""
    if pd.isna(s):
        return np.nan
    s = str(s)
    if 'RM' not in s:
        return np.nan
    # Handles both 'RM 208,000' and '-4% RM 105,800'
    s = s.split('RM')[-1].replace(',', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def clean_monthly(s):
    """Strip RM / month and return float; blanks return NaN."""
    if pd.isna(s):
        return np.nan
    s_str = str(s).strip()
    if s_str == '' or s_str == 'nan':
        return np.nan
    if 'RM' not in s_str:
        return np.nan
    s_str = s_str.split('RM')[-1].replace('/ month', '').replace(',', '').strip()
    try:
        return float(s_str)
    except ValueError:
        return np.nan


def parse_mileage(s):
    """Extract lower-bound km from range string like '90 - 95K KM'."""
    if pd.isna(s):
        return 0
    s = str(s)
    parts = s.split(' - ')
    try:
        val = int(parts[0].strip())
    except ValueError:
        return 0
    if 'K' in s.upper():
        val *= 1000
    return val

In [ ]:
# seller_type: extract label before boilerplate ('Sales Agent This Trusted Dealer...')
df['seller_type'] = df['seller_type'].str.split(' This').str[0].str.strip()

# price and monthly_payment
df['price']           = df['price'].apply(clean_price)
df['monthly_payment'] = df['monthly_payment'].apply(clean_monthly)

# mileage -> mileage_km
df['mileage_km'] = df['mileage'].apply(parse_mileage)
df.drop(columns=['mileage'], inplace=True)

# location: 'Selangor, Shah Alam' -> state='Selangor', city='Shah Alam'
loc_split   = df['location'].str.split(', ', n=1, expand=True)
df['state'] = loc_split[0].str.strip()
df['city']  = loc_split[1].fillna('N/A').str.strip()
df.drop(columns=['location'], inplace=True)

# year: ensure integer
df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

# Fill remaining blank strings with 'N/A'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].apply(lambda x: 'N/A' if pd.isna(x) or str(x).strip() == '' else x)

print(f'Cleaned shape: {df.shape}')
print('\nColumn dtypes after cleaning:')
print(df.dtypes)

## 4. Verify Cleaned Data

In [6]:
print('Sample of cleaned data:')
display(df.head(5))

print('\nNull values after cleaning:')
print(df.isnull().sum())

print('\nUnique values in key categorical columns:')
for col in ['transmission', 'seller_type', 'condition']:
    print(f'  {col}: {sorted(df[col].unique())}')

Sample of cleaned data:


,id,title,year,price,monthly_payment,transmission,seller_type,condition,mileage_km,state,city
0,18990060,2026 BMW 218 1.5 Gran Sport Coupe - FREE WARRA...,2026,208000.0,2697.0,Automatic,Sales Agent,USED,0,Selangor,Shah Alam
1,18974192,2017 Toyota Corolla Altis 1.8 G Sedan,2017,55800.0,723.0,Automatic,Dealer,USED,90000,Perak,Teluk Intan
2,18990035,2020 Nissan Serena 2.0 S-Hybrid High-Way Star ...,2020,69800.0,905.0,Automatic,Sales Agent,USED,85000,Selangor,Seri Kembangan
3,18990057,2021 Porsche 911 3.0 Carrera S Coupe,2021,638000.0,8271.0,Automatic,Sales Agent,RECON,15000,Kuala Lumpur,Cheras
4,18754783,2022 Mercedes-Benz G400 2.9 D AMG Line SUV - D...,2022,515000.0,6677.0,Automatic,Sales Agent,RECON,25000,Selangor,Petaling Jaya



Null values after cleaning:
id                   0
title                0
year                 0
price               69
monthly_payment    174
transmission         0
seller_type          0
condition            0
mileage_km           0
state                0
city                 0
dtype: int64

Unique values in key categorical columns:
  transmission: ['Automatic', 'Manual']
  seller_type: ['Broker', 'Dealer', 'N/A', 'Private', 'Private Verified Verified Private Seller had been verified by Carlist.my with IC, Car Registration Card and valid car information showing on this ad.', 'Sales Agent']
  condition: ['NEW', 'RECON', 'USED']


In [7]:
print('Price statistics (RM):')
print(df['price'].describe().apply(lambda x: f'{x:,.0f}'))

print('\nTop 5 states by listing count:')
print(df['state'].value_counts().head())

Price statistics (RM):
count       99,949
mean       216,219
std        310,118
min              2
25%         59,800
50%        142,999
75%        253,000
max      4,999,999
Name: price, dtype: str

Top 5 states by listing count:
state
Kuala Lumpur    38062
Selangor        35276
Johor           17733
Perak            3298
Penang           2937
Name: count, dtype: int64


## 5. Save Cleaned CSV

In [8]:
df.to_csv(CLEANED_CSV, index=False)

size_mb = os.path.getsize(CLEANED_CSV) / 1024**2
print(f'Saved {len(df):,} rows to {CLEANED_CSV}')
print(f'File size: {size_mb:.1f} MB')

Saved 100,018 rows to carlist_cleaned.csv
File size: 13.6 MB
